# Portfolio Optimization Across Constraint Regimes
### A Linear-Programming Approach with Monte-Carlo and CVaR Extensions

**MSDS 460 — Decision Analytics · Northwestern · Dr. Kline**  
**Team:** Rachel Raia · Jung Suh · Scott Keighley

---

> **Mock-report scaffold.** Each section below shows the figures we plan to include and the key facts that need to appear in the prose. Rachel: draft the narrative around the bullets, then compile to HTML with `jupyter nbconvert --to html notebooks/Project_Overview.ipynb`. Bullets marked **fact:** come straight from the source notebook results; bullets marked **frame:** are interpretive points to weave in.

## How to read this scaffold

- 📊 Figures are pulled from the four working notebooks and live in `notebooks/figures/`.
- 📝 Each section ends with a **DRAFT** placeholder where Rachel inserts the narrative paragraph(s).
- 🔁 Numbers in bullets are pulled from current notebook outputs — if a notebook is re-run, double-check the values before publishing.

## Source notebooks

| Tag | Notebook | Role |
|-----|----------|------|
| NB1 | `Final_Project_Portfolio_Optimization.ipynb` | Data pipeline |
| NB2 | `Final_Project_Portfolio_Optimization_LP.ipynb` | LP single-window model |
| NB3 | `Final_Project_Portfolio_Optimization_Walk_Forward.ipynb` | LP walk-forward backtest |
| NB4 | `Monte_Carlo_Portfolio_Optimization.ipynb` | Monte Carlo, QP, CVaR |

## 1 · Executive Summary

**Key points (draft a 1-paragraph abstract from these):**

- **frame:** Question — can a constrained linear program deliver risk-adjusted outperformance against a naive equal-weight benchmark across a full 7-year out-of-sample window (2019-06 → 2026-02)?
- **fact:** LP terminal wealth **$5.98** vs. equal-weight **$3.75** on $1 invested (CAGR **29.7%** vs. **21.1%**).
- **fact:** LP Sharpe **1.208** vs. EW Sharpe **1.264** — equal-weight is *more* risk-efficient despite lower absolute return.
- **fact:** LP max drawdown **−36.97%** vs. EW **−19.91%** — concentration cost paid in 2022.
- **fact:** Across QP (max-Sharpe / min-vol), CVaR, and equal-weight, OOS Sharpe rankings show equal-weight is the hardest benchmark to beat.
- **frame:** Estimation risk dominates — in-sample expected returns correlate **−0.47** with realized returns across periods.
- **frame:** Position cap is the most actionable lever; sector cap is slack at our current settings.

### DRAFT NEEDED — Rachel
_Single paragraph (≈150 words) framing the question and headline result. Lead with the trade-off: LP wins on absolute return but loses on Sharpe and drawdown. The interesting story is **why**._

## 2 · Data & Asset Universe

**Key points:**

- **fact:** 30 large-cap S&P 500 equities across 10–11 GICS sectors.
- **fact:** Yahoo Finance monthly adjusted close, **2016-05 → 2026-05** (120 observations, 119 monthly returns).
- **fact:** Walk-forward params — 36-month training window, 3-month rebalance for the LP backtest; 1-month for the QP backtest.
- **frame:** Long enough to span multiple regimes (COVID crash & rally, 2022 tech drawdown, 2023-25 mega-cap concentration) but short enough that survivorship bias matters — note in caveats.

### 2.1 Normalized price growth (10-year)

![Normalized price growth across the 30-asset universe](figures/01_data_fig01.png)

- **frame:** Wide dispersion of outcomes; NVDA, AVGO, AAPL, MSFT pull the right tail.
- **frame:** Lower spaghetti shows energy / consumer-staples cluster.

### 2.2 Correlation structure

![Correlation heatmap of monthly returns](figures/01_data_fig03.png)

- **fact:** Mean off-diagonal correlation is moderate — typical large-cap co-movement.
- **frame:** Diversification benefit limited by sector clustering, especially in tech.

### DRAFT NEEDED — Rachel
_Two short paragraphs: (1) data provenance and cleaning approach, (2) what the EDA tells us about the diversification opportunity set. Cite Jung's NB1 for the cleaning procedure._

## 3 · Methodology

**Three formulations are tested:**

### 3.1 Linear Program (PuLP / CBC)

$$
\begin{aligned}
\max_{w} \quad & \mu^{\top} w - c \cdot \|w - w_{\text{prev}}\|_1 \\
\text{s.t.} \quad & \mathbf{1}^{\top} w = 1, \quad 0 \le w_i \le w_{\max} \\
& \sum_{i \in s} w_i \le s_{\max} \quad \forall \text{ sector } s \\
& \|w - w_{\text{prev}}\|_1 \le \tau_{\max}
\end{aligned}
$$

- **fact:** $w_{\max} = 0.15$, $s_{\max} = 0.40$, $\tau_{\max} = 0.50$, transaction cost $c = 10$ bps.
- **frame:** L1 turnover and absolute-value transaction cost are linearized with auxiliary variables — keeps the model an LP, solvable with CBC in milliseconds.

### 3.2 Quadratic Program (Markowitz, SLSQP)

- Max-Sharpe and min-vol variants on the same sample covariance matrix.
- Position cap 10%, sector cap 30%.

### 3.3 CVaR LP (Rockafellar & Uryasev 2000)

- Minimize expected loss in the worst α-tail; reformulated as LP via auxiliary variables.
- Same position/sector constraints as the LP.

### 3.4 Walk-forward protocol (unified)

- Rolling 36-month training window → solve optimizer → hold weights for `REBAL_MONTHS` → roll forward.
- OOS window June 2019 → Feb 2026 (83 months; 27 quarterly periods for LP, 83 monthly periods for QP).
- Benchmarks: equal-weight 30-asset, SPY.

### DRAFT NEEDED — Rachel
_Half a page explaining why three formulations are useful — LP for tractability and interpretability, QP for the classical risk-return trade-off, CVaR for tail-risk focus. Note that all three share the same data, constraints, and walk-forward scaffold so differences in results are attributable to the objective function._

## 4 · Single-Window LP Analysis

_Source: NB2. Goal — build intuition for what the LP does before scaling to the full backtest._

### 4.1 Optimal allocation, first training window

![LP allocation bar chart by sector](figures/02_lp_fig01.png)

- **fact:** Optimizer concentrates at the **15% position cap** on a handful of names (high-momentum tech).
- **frame:** Linear objective means corner solutions — without caps, the LP would put 100% on the single highest-mean asset.

### 4.2 Sector allocation

![Sector allocation pie and bar](figures/02_lp_fig02.png)

- **fact:** Tech sector hits the 40% sector cap; consumer-discretionary and financials fill the remainder.
- **frame:** Sector cap is the **only** constraint preventing total tech concentration in this window.

### 4.3 Sensitivity — position cap

![Sensitivity to position cap parameter](figures/02_lp_fig03.png)

- **fact:** Relaxing the position cap from 15% → 100% gains **≈1.88%** annualized expected return.
- **frame:** That 1.88% is the *in-sample* cost of diversification — actual OOS cost is much smaller (or negative) because mean estimates are noisy.

### 4.4 Sensitivity — sector cap

![Sensitivity to sector cap parameter](figures/02_lp_fig04.png)

- **fact:** Sector cap is **non-binding** across most of the sweep — expected return flat.
- **frame:** Implies the sector cap is redundant given the position cap; useful as a guardrail but does not bite in normal conditions.

### DRAFT NEEDED — Rachel
_One paragraph on what a single-window solve looks like, then one paragraph on the constraint sensitivity insight (which constraint costs us return, which one doesn't). The sensitivity finding is genuinely interesting — most LP write-ups never quantify it._

## 5 · Walk-Forward Results

### 5.1 LP — quarterly rebalance (NB3)

#### Cumulative performance

![LP vs equal-weight cumulative return](figures/03_wf_fig01.png)

- **fact:** LP terminal wealth **$5.98** vs. EW **$3.75** (29.7% vs. 21.1% CAGR).
- **fact:** Outperformance concentrated in the 2020-21 COVID rally; LP is roughly flat-to-down relative to EW in 2022.

#### Drawdown

![Drawdown of LP vs equal-weight](figures/03_wf_fig07.png)

- **fact:** LP max drawdown **−36.97%** in 2022 vs. EW **−19.91%**.
- **frame:** Concentration risk made explicit — the same constraint structure that delivers the upside delivers the downside.

#### Composition over time

![Weight heatmap over walk-forward periods](figures/03_wf_fig05.png)

- **fact:** Holdings converge from 23 → 7 assets within the first four quarters.
- **frame:** Regime rotation visible — NVDA grows, AMZN exits, energy briefly appears in 2022.

#### Turnover and transaction cost

![Turnover and transaction costs over time](figures/03_wf_fig04.png)

- **fact:** Mean quarterly turnover **29.75%**; cumulative transaction cost **80 bps** over 7 years.
- **frame:** Low-friction strategy in practice; the turnover cap rarely binds.

#### In-sample vs. realized return

![Realized vs expected return by period](figures/03_wf_fig02.png)

- **fact:** Pearson correlation between in-sample expected return and realized return across periods is **−0.47**.
- **frame:** *Negative* correlation — the LP's confidence is anti-predictive at the period level. This is the estimation-risk thread of the report.

#### Assets held

![Number of assets held over time](figures/03_wf_fig06.png)

- **fact:** Converges to **7 assets** by mid-2020, stable thereafter.
- **frame:** Effective number of bets is far smaller than the 30-asset universe — diversification mostly forfeited.

### DRAFT NEEDED — Rachel (5.1)
_Three paragraphs: (1) headline result and equity curve; (2) drawdown / concentration trade-off in 2022; (3) the estimation-risk story from the expected-vs-realized chart. The third is the most important conceptual point in the report._

### 5.2 QP and CVaR — monthly rebalance (NB4)

#### Monte Carlo cloud and efficient frontier

![Monte Carlo cloud with efficient frontier overlay](figures/04_mc_fig01.png)

- **fact:** 50,000 random long-only portfolios sampled; frontier traced as upper envelope.
- **frame:** Provides visual intuition for the feasible set before introducing constraints.

#### Constrained mean-variance solution

![Constrained max-Sharpe and min-vol points on the MC cloud](figures/04_mc_fig02.png)

- **fact:** Max-Sharpe and min-vol solutions sit interior to the unconstrained frontier — the cost of position/sector caps is visible.
- **frame:** QP corners are softened by the quadratic objective — fewer assets at the cap than the LP.

#### Composition over the walk-forward

![Max-Sharpe walk-forward weight composition](figures/04_mc_fig03.png)

- **frame:** QP holdings rotate more smoothly than LP holdings — quadratic penalty discourages corner solutions.

#### SPY benchmark

![Strategy vs SPY cumulative return](figures/04_mc_fig04.png)

- **fact:** Equal-weight 30-asset is a *harder* benchmark than SPY across this window — beating SPY is not sufficient to claim alpha.

### DRAFT NEEDED — Rachel (5.2)
_Two paragraphs: (1) what the MC cloud and frontier show; (2) how QP behaves differently from LP — smoother weights, less corner-seeking. End with the SPY-vs-EW point: choice of benchmark matters._

### 5.3 Head-to-head comparison ⚠️ TO BE BUILT

> **Gap to close.** NB3 (LP) and NB4 (QP, CVaR) never report on the **same** backtest protocol. Before submission we need one consolidated table:

| Strategy | CAGR | Vol | Sharpe | Calmar | Max DD | Turnover | Hit rate |
|---|---|---|---|---|---|---|---|
| LP (quarterly) | _29.7%_ | — | _1.208_ | — | _−37.0%_ | _29.8%_ | _63%_ |
| QP max-Sharpe (monthly) | — | — | _≈1.10_ | — | — | — | — |
| QP min-vol (monthly) | — | — | — | — | — | — | — |
| CVaR LP | — | — | — | — | — | — | — |
| Equal-weight | _21.1%_ | — | _1.264_ | — | _−19.9%_ | 0% | _50%_ |
| SPY | — | — | — | — | — | 0% | — |

- **Action item:** Either (a) re-run NB3's metrics function on NB4's weights, or (b) re-run all strategies on a single shared protocol (quarterly or monthly — pick one).
- **Frame for prose:** This table is the punch line. Rachel — reserve a paragraph here for the comparative discussion once the numbers are populated.

## 6 · Robustness

### 6.1 Historical bootstrap

![Bootstrap simulation paths](figures/04_mc_fig05.png)

- **fact:** 1,000 bootstrap paths over a 120-month horizon, resampling from the 83-month OOS returns.
- **frame:** Spread of terminal wealth quantifies sequence-of-returns risk under the assumption that future months resemble past OOS months.

### 6.2 Regime-conditional bootstrap

![High-vol vs low-vol regime bootstrap distributions](figures/04_mc_fig06.png)

- **fact:** Resampling stratified by 3-month rolling volatility shows materially different terminal-wealth distributions across regimes.
- **frame:** Min-vol strategy outperforms max-Sharpe in the high-vol regime by a wide margin.

### 6.3 Parameter sensitivity

![OOS Sharpe across parameter sweeps](figures/04_mc_fig07.png)

- **fact:** Sweeps over `train_months`, `position_cap`, `sector_cap`, `mu_shrinkage`.
- **fact:** Best position cap is **5–7.5%**; best training window is **36–48 months**.

### 6.4 Tornado view

![Tornado chart of OOS Sharpe sensitivity](figures/04_mc_fig08.png)

- **fact:** Position cap dominates — spread of **0.29** in OOS Sharpe across the sweep.
- **fact:** Sector cap moves OOS Sharpe by only **0.03** — effectively redundant.
- **frame:** If we had to defend a single parameter choice, it would be the position cap.

### DRAFT NEEDED — Rachel
_One paragraph on the bootstrap (what it tells us about path dependence), one paragraph on the sensitivity (which knobs matter, which don't). The tornado is the single most defensible robustness exhibit in the report._

## 7 · Discussion

**Three threads to weave together:**

1. **Estimation risk.** In-sample mean estimates are anti-predictive period-by-period (ρ = −0.47). The LP's confidence in its objective is unwarranted at the rebalance frequency. _Cite the realized-vs-expected chart in §5.1._
2. **Concentration vs. diversification.** LP captures regime upside (COVID rally) at the cost of regime downside (2022 tech drawdown). Equal-weight pays a return premium for path-stability. _Cite the drawdown chart and the head-to-head table in §5.3._
3. **Constraint design.** The position cap is the load-bearing constraint. The sector cap is a guardrail that doesn't bind. Turnover cap rarely binds either. _Cite tornado in §6.4._

**Caveats to acknowledge:**

- Survivorship bias — universe defined on today's S&P 500 membership.
- Single market regime — 2019-2026 is a specific period; bull-biased.
- Transaction cost modeled as constant 10 bps — real costs depend on liquidity and order size.
- No short-selling, no leverage — long-only fully-invested only.

### DRAFT NEEDED — Rachel
_The discussion is the highest-effort prose section. Aim for ~500 words organized around the three threads. The estimation-risk thread is the academically interesting one for an MSDS audience — give it the most weight._

## 8 · Conclusion

**Key points:**

- **frame:** Linear programming gives an interpretable, fast, constraint-rich allocation tool.
- **frame:** Constraints matter more than the objective function — equal-weight wins on Sharpe because it sidesteps estimation error.
- **frame:** Robustness analysis (bootstrap, sensitivity) is essential — a single backtest path is not a result.
- **frame:** Future work — shrinkage estimators for μ, regime-aware models, more frequent rebalancing with realistic friction.

### DRAFT NEEDED — Rachel
_Short — ~150 words. Restate the question, the headline result, and the one-line lesson._

---

## Appendix — author contributions

- **Scott Keighley:**  Data pipeline, LP formulation, walk-forward backtest.
- **Jung Suh:** Data pipeline, Monte Carlo, QP / CVaR, robustness analysis.
- **Rachel Raia:** Report integration, narrative, head-to-head comparison.

